# Gate 1 — Reconstruction Equivalence Test

**Purpose:** Verify that FACodecAdapter produces bit-identical (up to floating-point)
reconstructions compared to the upstream FAcodec encode→quantizer→decode pipeline.

**Numeric criteria:**
- max |upstream_recon - adapter_recon| < 1e-4
- SNR(upstream, adapter) > 60 dB
- len(upstream) == len(adapter) exactly

**Stop condition:** If ANY criterion fails on ANY sample → abort and report FAIL.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# 2. Setup paths and run environment manifest
import os, sys, json, time, subprocess, types, warnings, hashlib, getpass
from pathlib import Path

warnings.simplefilter("ignore")

# ── Workspace ──
ACCENTEDGE_DIR = "/content/accentedge"
FA_CODEC_DIR   = "/content/FAcodec"
GATE_DIR       = "/content/gate1_artifacts"
DRIVE_BASE     = "/content/drive/MyDrive/accentedge/runs"
SAMPLE_RATE    = 24000
HOP_LENGTH     = 300
FPS            = 80  # SAMPLE_RATE // HOP_LENGTH

def run(cmd, desc="", check=True, timeout=120):
    print(f"\n>>> {desc or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = r.stdout.strip()
    if out:
        print(out[:500])
    if check and r.returncode != 0:
        print(f"FAILED: {r.stderr[:500]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return r

# ── GPU check ──
r = run("nvidia-smi --query-gpu=name --format=csv,noheader", "GPU", check=False)
gpu_name = r.stdout.strip()
print(f"GPU: {gpu_name}")

# ── Run environment manifest ──
manifest_path = f"{GATE_DIR}/environment.json"
if os.path.exists(manifest_path):
    print(f"Manifest already exists at {manifest_path}")
    with open(manifest_path) as f:
        env = json.load(f)
else:
    print("\nRecording environment manifest...")
    sys.path.insert(0, FA_CODEC_DIR)
    sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")
    os.chdir(FA_CODEC_DIR)
    env = {}
    env["gpu_name"] = gpu_name
    env["python_version"] = sys.version.split()[0]
    import torch
    env["torch_version"] = torch.__version__
    env["cuda_version"] = torch.version.cuda if torch.cuda.is_available() else "N/A"
    env["cuda_available"] = torch.cuda.is_available()
    r = run("cd /content/accentedge && git rev-parse HEAD", "accentedge SHA", check=False)
    env["accentedge_git_sha"] = r.stdout.strip()
    r = run("cd /content/accentedge && git rev-parse --abbrev-ref HEAD", "accentedge branch", check=False)
    env["accentedge_git_branch"] = r.stdout.strip()
    r = run("cd /content/FAcodec && git rev-parse HEAD", "FAcodec SHA", check=False)
    env["facodec_upstream_revision"] = r.stdout.strip()
    from modules.commons import recursive_munch
    from hf_utils import load_custom_model_from_hf
    ckpt_path, config_path = load_custom_model_from_hf("Plachta/FAcodec")
    env["facodec_ckpt_path"] = ckpt_path
    env["facodec_ckpt_hash"] = hashlib.sha256(open(ckpt_path, "rb").read()).hexdigest()[:16]
    env["manifest_timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    os.makedirs(GATE_DIR, exist_ok=True)
    with open(f"{GATE_DIR}/environment.json", "w") as f:
        json.dump(env, f, indent=2)
    print(f"Manifest saved: {GATE_DIR}/environment.json")

git_sha = env.get("accentedge_git_sha", "unknown")
print(f"\naccentedge SHA: {git_sha}")

In [ ]:
# 3. Clone repos (idempotent)
run("test -d /content/FAcodec || git clone https://github.com/Plachtaa/FAcodec.git /content/FAcodec",
    "clone FAcodec", check=False)
run("test -f /content/FAcodec/modules/__init__.py || touch /content/FAcodec/modules/__init__.py",
    "modules init", check=False)
run("test -d /content/accentedge || git clone --depth 1 https://github.com/yagami009/accentedge.git /content/accentedge",
    "clone accentedge", check=False)

In [ ]:
# 4. Install dependencies
!pip install -q numpy soundfile librosa scipy jiwer pyyaml einops \
    huggingface-hub phonemizer speechbrain torchaudio faster-whisper \
    pytest pyworld munch plotly ipykernel
print("Dependencies installed")

In [ ]:
# 5. Path setup
sys.path = [p for p in sys.path if "/content" not in p]
sys.path.insert(0, FA_CODEC_DIR)
sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")
os.environ["PYTHONPATH"] = FA_CODEC_DIR + "/modules:" + os.environ.get("PYTHONPATH", "")
os.chdir(FA_CODEC_DIR)
print(f"PYTHONPATH={os.environ['PYTHONPATH']}")

In [ ]:
# 6. Run Gate 1 reconstruction test
import importlib.util, traceback

spec = importlib.util.spec_from_file_location("gate1_recon", f"{ACCENTEDGE_DIR}/scripts/gate1_reconstruction.py")
mod  = importlib.util.module_from_spec(spec)

try:
    spec.loader.exec_module(mod)
    mod.main()
except SystemExit as e:
    if e.code != 0:
        print("\n" + "="*60)
        print("GATE 1 RESULT: FAIL")
        print("="*60)
        raise
except Exception as e:
    print(f"\nEXCEPTION: {e}")
    traceback.print_exc()
    raise

In [ ]:
# 7. Check results & print PASS/FAIL
import json, shutil

metrics_path = f"{GATE_DIR}/reconstruction_metrics.json"
if not os.path.exists(metrics_path):
    print("ERROR: reconstruction_metrics.json not found — Gate 1 did not complete.")
    raise FileNotFoundError(metrics_path)

with open(metrics_path) as f:
    results = json.load(f)

all_pass = results.get("overall_pass", False)
print("\n" + "="*60)
print("GATE 1 FINAL RESULT")
print("="*60)
print(f"  Samples tested:    {results.get('samples', []) and len(results['samples'])}")
print(f"  Max abs diff:      {results.get('max_abs_diff_overall', 'N/A'):.2e}  (threshold: 1e-4)")
print(f"  Min SNR:           {results.get('min_snr_db_overall', 'N/A'):.2f} dB  (threshold: 60 dB)")
print(f"  Overall:           {'PASS' if all_pass else 'FAIL'}")

# Per-sample table
if results.get("samples"):
    print(f"\n{'Idx':<4} {'File':<35} {'MaxDiff':>12} {'SNR(dB)':>10} {'LenOK':>6} {'Result':>8}")
    print(f"{'─'*4} {'─'*35} {'─'*12} {'─'*10} {'─'*6} {'─'*8}")
    for s in results["samples"]:
        status = "PASS" if s["passed_all"] else "FAIL"
        print(f"{s['sample_idx']:<4} {s['filename']:<35} "
              f"{s['max_abs_diff']:>12.2e} {s['snr_db']:>10.2f} "
              f"{'Y' if s['length_match'] else 'N':>6} {status:>8}")

# ── Save to Drive ──
drive_out = f"{DRIVE_BASE}/{git_sha}/gate1"
os.makedirs(drive_out, exist_ok=True)
for fname in ["reconstruction_metrics.json", "latent_stats.json", "environment.json"]:
    src = f"{GATE_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_out}/{fname}")

# Copy WAV artifacts
for subdir in ["source", "upstream_reconstruction", "adapter_reconstruction"]:
    d = f"{GATE_DIR}/{subdir}"
    if os.path.isdir(d):
        dst = f"{drive_out}/{subdir}"
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(d):
            if f.endswith(".wav"):
                shutil.copy2(f"{d}/{f}", f"{dst}/{f}")

print(f"\nArtifacts saved to: {drive_out}")

if not all_pass:
    raise RuntimeError("GATE 1 FAILED — aborting pipeline.")

In [ ]:
# 8. Pipeline status
print("\nGate 1 complete.")
print(f"Run directory: {DRIVE_BASE}/{git_sha}/gate1")